In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import requests
import os
import time
import csv

### Scraping webpages that contain motion voting results

In [6]:
# Set up Chrome WebDriver
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)

# Target URL (Replace with actual URL)
base_url = "https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen?fromdate=&todate=2012-03-17&fld_prl_kamerstuk=Stemmingsuitslagen&fld_tk_categorie=Kamerstukken&qry=*&srt=date%3Adesc%3Adate&sta=1"
driver.get(base_url)

# Allow page to load
time.sleep(2)

# Store extracted links
extracted_links = []

while True:
    try:
        # Find all card containers that might contain <h4> and <a> elements
        card_containers = driver.find_elements(By.CSS_SELECTOR, "div.m-card__content")

        for container in card_containers:
            try:
                # Locate the <a> tag inside the container
                a_tag = container.find_element(By.CSS_SELECTOR, "a.h-link-inverse")
                href = a_tag.get_attribute("href")

                # Save only if the link is not already stored
                if href and href not in extracted_links:
                    extracted_links.append(href)
                    print("Extracted link:", href)
            except Exception:
                print("No <a> tag found in this container.")

        # Try to locate the "Next" button
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "li.m-pager__item.m-pager__item--next a")
            next_page_url = next_button.get_attribute("href")

            if next_page_url:
                print("Navigating to next page:", next_page_url)
                driver.get(next_page_url)
                time.sleep(2)  # Allow new page to load
            else:
                break  # Stop if no valid next page is found

        except Exception:
            print("No more pages to navigate.")
            break  # Exit loop when no "Next" button exists

    except Exception as e:
        print("Error:", e)
        break

# Save results to a CSV file
with open("scraped_links_I_think_final_2012_down.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Links"])
    for link in extracted_links:
        writer.writerow([link])

print("All links saved to scraped_links.csv")

# Close the browser
driver.quit()

No <a> tag found in this container.
Extracted link: https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen/detail?id=2012P03871&did=2012P03871
No <a> tag found in this container.
Extracted link: https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen/detail?id=2012P03868&did=2012P03868
No <a> tag found in this container.
Extracted link: https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen/detail?id=2012P03869&did=2012P03869
No <a> tag found in this container.
Extracted link: https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen/detail?id=2012P03898&did=2012P03898
No <a> tag found in this container.
Extracted link: https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen/detail?id=2012P03901&did=2012P03901
No <a> tag found in this container.
Extracted link: https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen/detail?id=2012P03899&did=2012P03899
No <a> tag found in this container.
Extracted link: https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen/detail?

### Scraping specific motion links

In [16]:
# Set up Chrome WebDriver
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)

# Input and output CSV files
input_csv = "scraped_links_final_2014-2025.csv"   # CSV file containing URLs to visit
output_csv = "motion_links_2014-2025.csv" # CSV file containing URLs to visit

# Read URLs from input CSV file
with open(input_csv, mode="r", encoding="utf-8") as file:
    reader = csv.reader(file)
    urls = [row[0] for row in reader]  # Extract first column containing URLs

# Prepare list to store extracted data
data = []

for url in urls:
    try:
        driver.get(url)  # Visit the URL
        time.sleep(2)  # Wait for page to load

        # Locate all <p> elements with class "u-mt-8"
        p_elements = driver.find_elements(By.CSS_SELECTOR, "p.u-mt-8")
        extracted_texts = []

        for p_element in p_elements:
            try:
                # Extract text from <span class="u-font-bold">
                span_element = p_element.find_element(By.CSS_SELECTOR, "span.u-font-bold")
                extracted_text = span_element.text.strip()
                extracted_texts.append(extracted_text)
            except:
                continue  # Skip if <span> not found

        # Locate all <a> elements under <h3 class="m-card__title">
        link_elements = driver.find_elements(By.CSS_SELECTOR, "h3.m-card__title a")
        hrefs = [link.get_attribute("href") for link in link_elements]

        # Ensure extracted_texts and hrefs match correctly
        for extracted_text, href_value in zip(extracted_texts, hrefs):
            if extracted_text in ["Aangenomen.", "Verworpen."]:
                data.append([url, extracted_text, href_value])

    except Exception as e:
        print(f"Error processing {url}: {e}")

# Save extracted data to a CSV file only if there is data to save
if data:
    with open(output_csv, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Source URL", "Text", "Extracted Href"])  # CSV headers
        writer.writerows(data)

    print(f"Data saved to {output_csv}")
else:
    print("No relevant data found. No CSV file was created.")

# Close the browser
driver.quit()

Error processing Links: Message: invalid argument
  (Session info: chrome=135.0.7049.85)
Stacktrace:
	GetHandleVerifier [0x0065D363+60275]
	GetHandleVerifier [0x0065D3A4+60340]
	(No symbol) [0x0049056E]
	(No symbol) [0x00480ADE]
	(No symbol) [0x0047F1D5]
	(No symbol) [0x0047F91B]
	(No symbol) [0x0049407E]
	(No symbol) [0x0051F2F7]
	(No symbol) [0x004FD08C]
	(No symbol) [0x0051E6EB]
	(No symbol) [0x004FCE86]
	(No symbol) [0x004CC623]
	(No symbol) [0x004CD474]
	GetHandleVerifier [0x008A8FE3+2467827]
	GetHandleVerifier [0x008A45E6+2448886]
	GetHandleVerifier [0x008BF80C+2560028]
	GetHandleVerifier [0x00673DF5+153093]
	GetHandleVerifier [0x0067A3BD+179149]
	GetHandleVerifier [0x00664BB8+91080]
	GetHandleVerifier [0x00664D60+91504]
	GetHandleVerifier [0x0064FA10+4640]
	BaseThreadInitThunk [0x75735D49+25]
	RtlInitializeExceptionChain [0x772DCF0B+107]
	RtlGetAppContainerNamedObjectPath [0x772DCE91+561]

Data saved to motion_links_2014-2025.csv


### Subsetting the motion_links into 10 subsets

In [18]:
# Loading CSV-file with extracted links
motion_links = pd.read_csv("motion_links_2014-2025.csv")

# Setting subset size
chunk_size = 4000

# Creating output folder
output_dir = "subset_files_motion_links"
os.makedirs(output_dir, exist_ok=True)

# Creating subsets of the CSV-file
for i in range(0, len(motion_links), chunk_size):
    chunk = motion_links.iloc[i:i + chunk_size]
    chunk_filename = os.path.join(output_dir, f"subset_{i // chunk_size + 1}.csv")
    chunk.to_csv(chunk_filename, index=False)
    print(f"Saved {chunk_filename} with {len(chunk)} rows.")

Saved subset_files_motion_links\subset_1.csv with 4000 rows.
Saved subset_files_motion_links\subset_2.csv with 4000 rows.
Saved subset_files_motion_links\subset_3.csv with 4000 rows.
Saved subset_files_motion_links\subset_4.csv with 4000 rows.
Saved subset_files_motion_links\subset_5.csv with 4000 rows.
Saved subset_files_motion_links\subset_6.csv with 4000 rows.
Saved subset_files_motion_links\subset_7.csv with 4000 rows.
Saved subset_files_motion_links\subset_8.csv with 4000 rows.
Saved subset_files_motion_links\subset_9.csv with 4000 rows.
Saved subset_files_motion_links\subset_10.csv with 2350 rows.


### Scraping motion outcomes

In [ ]:
# Set up Chrome WebDriver with optimizations
service = Service(ChromeDriverManager().install())
options = Options()
options.add_argument("--disable-gpu")
options.add_argument("--blink-settings=imagesEnabled=false")  # Disable images
options.add_argument("--disable-extensions")

driver = webdriver.Chrome(service=service, options=options)

# Input CSV (contains extracted hrefs) and output CSV (detailed motion data)
input_csv = "subset_[nr].csv"  # File with the hrefs
output_csv = "motion_details_subset_[nr].csv"  # File to store extracted motion details

# Read URLs from input CSV file (skip header)
with open(input_csv, mode="r", encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader)  # Skip the header
    urls = [row[2] for row in reader if row[2] != "N/A"]  # Extract href column

# Prepare list to store extracted data
all_motion_data = []

for url in urls:
    try:
        driver.get(url)
        time.sleep(2)  # Wait for the page to load

        # Extract the motion title (h1 text)
        try:
            h1_element = WebDriverWait(driver, 5).until(EC.visibility_of_element_located((By.TAG_NAME, "h1")))
            motion_text = h1_element.text.strip()
        except:
            motion_text = "N/A"

        # Extract petitioner(s)
        try:
            petition_section = driver.find_elements(By.CSS_SELECTOR, "div.u-mt-8 ul.m-list span.m-list__label")
            petitioner_texts = [item.text.strip() for item in petition_section]
            petitioners = ", ".join(petitioner_texts) if petitioner_texts else "N/A"
        except:
            petitioners = "N/A"

         # Extract date
        try:
            first_li = driver.find_element(By.CSS_SELECTOR, "aside.t-grid__col.sm\\:u-pr-0.sm\\:u-pl-0 ul.u-mt-0.u-bg-background.u-p-3.m-list.m-list--has-dividers.m-list--divider-style-white > li.m-list__item")
            label_span = first_li.find_element(By.CSS_SELECTOR, "span.m-list__label")
            date_text = label_span.text.strip()
            date = date_text if date_text else "N/A"

        except Exception as e:
            date = "N/A"

        # Click the button to reveal the voting table
        try:
            button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.m-toggler__handler")))
            driver.execute_script("arguments[0].click();", button)
            time.sleep(2)  # Allow content to expand
        except:
            print(f"Button not found for {url}")

        # Extract voting results
        voting_results = []
        try:
            tbody = WebDriverWait(driver, 5).until(EC.visibility_of_element_located((By.CSS_SELECTOR, "div.m-toggler__body div.u-overflow-x-auto table.h-table-bordered tbody")))

            for row in tbody.find_elements(By.TAG_NAME, "tr"):
                columns = row.find_elements(By.TAG_NAME, "td")
                if columns:  # Only process rows with <td>
                    row_text = [col.text.strip() for col in columns]
                    voting_results.append(" | ".join(row_text))

        except:
            voting_results.append("No voting results found")

         # Exctracting the full description of the motion
        try:
            # Wait for the button to be visible
            button = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.u-mt-8 button[data-modal-toggle='#document-html-modal']")))
            # Force click using JavaScript
            driver.execute_script("arguments[0].click();", button)

            # Wait for the modal to appear and extract the text
            description_content = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, "div.m-modal__content")))
            description_text = description_content.text.strip()

        except Exception as e:
            print(f"Error: {e}")
        
        # Append extracted data
        all_motion_data.append([url, motion_text, description_text, petitioners, date, "; ".join(voting_results)])

    except Exception as e:
        print(f"Error processing {url}: {e}")

# Save extracted data to a CSV file
if all_motion_data:
    with open(output_csv, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Motion URL", "Motion Title", "Description Text", "Petitioner(s)", "Date", "Voting Results"])  # CSV headers
        writer.writerows(all_motion_data)

    print(f"Data saved to {output_csv}")
else:
    print("No relevant data found. No CSV file was created.")

# Close the browser
driver.quit()

Error: Message: 
Stacktrace:
	GetHandleVerifier [0x0065D363+60275]
	GetHandleVerifier [0x0065D3A4+60340]
	(No symbol) [0x004906F3]
	(No symbol) [0x004D8690]
	(No symbol) [0x004D8A2B]
	(No symbol) [0x00520EE2]
	(No symbol) [0x004FD0D4]
	(No symbol) [0x0051E6EB]
	(No symbol) [0x004FCE86]
	(No symbol) [0x004CC623]
	(No symbol) [0x004CD474]
	GetHandleVerifier [0x008A8FE3+2467827]
	GetHandleVerifier [0x008A45E6+2448886]
	GetHandleVerifier [0x008BF80C+2560028]
	GetHandleVerifier [0x00673DF5+153093]
	GetHandleVerifier [0x0067A3BD+179149]
	GetHandleVerifier [0x00664BB8+91080]
	GetHandleVerifier [0x00664D60+91504]
	GetHandleVerifier [0x0064FA10+4640]
	BaseThreadInitThunk [0x75735D49+25]
	RtlInitializeExceptionChain [0x772DCF0B+107]
	RtlGetAppContainerNamedObjectPath [0x772DCE91+561]

Data saved to motion_details_I_think.csv
